# Stable model reduction for linear variational inequalities — walkthrough

Niakh, Drouet, Ehrlacher, Ern, ESAIM: M2AN (2022).

Read `REPRODUCTION_NOTES.md` first. The 2-D test cases of §5 are **not**
implemented and no number here matches a number in the paper. One claim (`β^dec ≈ 0`)
does not reproduce; §6 of this notebook shows why, with a proof rather than a guess.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# The algorithms live in the shared library ~/rb_vi_shared, together with this
# paper's reference [9] in ~/rb_contact_cpg. _shared_path puts it on sys.path.
# Citations there carry a paper tag; everything imported below is [NDEE22], i.e.
# this paper. See ~/rb_vi_shared/README.md.
import _shared_path  # noqa: F401
from rb_vi_common import (Whitener, pod, supremizer_space, sigma_S, orth_union,
                          inf_sup, inf_sup_hf, boundedness_c_S,
                          cpg, mcpg, e_orth, pga, S_R_full)

from hf_model import ObstacleHF, training_set, validation_set

## 1. A parameter-dependent constraint

The whole paper turns on `b(µ;·,·)` depending on `µ`. Here the obstacle window
`ω(µ) = (c−r, c+r)` moves and resizes, so the collocation points `s_i(µ) = c + r·ŝ_i`
move — while `ψ̂` stays parameter-independent on the reference domain, as in §5.1.

In [ ]:
hf = ObstacleHF(n_elem=200, n_dual=60)
D_train, D_valid = training_set(n_r=8, n_c=8), validation_set(n=10)
U, L = hf.snapshots(D_train)
print(f"{hf.n_v} primal dofs, {hf.n_w} dual dofs, {len(D_train)} training params")
print(f"lambda >= 0: {L.min() >= 0}")
act = (L > 1e-10).sum(axis=0)
print(f"active constraints per mu: {act.min()}-{act.max()}")

# B(mu) really does change with mu
d = np.linalg.norm(hf.B(D_train[0]) - hf.B(D_train[-1]))
print(f"||B(mu_1) - B(mu_P)||_F = {d:.4f}   <- non-zero => S_R(mu) is mu-dependent")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
x = np.linspace(0, 1, hf.n_elem + 1)[1:-1]
for j in range(0, len(D_train), 12):
    ax[0].plot(x, U[:, j], lw=1.1)
    ax[1].plot(hf.s_hat, L[:, j], lw=1.1, label=f"r={D_train[j,0]:.2f}")
ax[0].set_title("primal $u(\\mu)$"); ax[0].set_xlabel("x")
ax[1].set_title("dual $\\lambda(\\mu)$ on the reference domain — all $\\geq 0$")
ax[1].set_xlabel("$\\hat s$"); ax[1].legend(fontsize=7)
for a in ax: a.grid(alpha=.3)
plt.tight_layout()

## 2. mCPG vs CPG — Algorithm 2

Remark 4.3: CPG appends the normalized snapshot; mCPG appends the residual after
removing the closest point of the cone that stays **below** `θ_q` in the cone order.
Eq. (41) measures how nearly orthogonal each new generator is to what came before.

In [ ]:
rc, rm = cpg(L, delta=2e-1), mcpg(L, delta=2e-1)
eo_c, eo_m = e_orth(rc.generators), e_orth(rm.generators)
m = min(len(eo_c), len(eo_m))
print(f"R:  CPG {rc.R}   mCPG {rm.R}")
print(f"mean e_orth:  CPG {eo_c[:m].mean():.4f}   mCPG {eo_m[:m].mean():.4f}")
print(f"C5 (e_orth^CPG <= e_orth^mCPG) on {100*np.mean(eo_c[:m] <= eo_m[:m]+1e-12):.0f}% of iterations")

def gram_cond(G):
    Gn = G / np.linalg.norm(G, axis=0, keepdims=True)
    return np.linalg.cond(Gn.T @ Gn)
print(f"Gram condition number:  CPG {gram_cond(rc.generators):.3e}   "
      f"mCPG {gram_cond(rm.generators):.3e}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(eo_c[:m], label="CPG", c="tab:red"); ax.plot(eo_m[:m], label="mCPG", c="tab:blue")
ax.set_xlabel("r"); ax.set_ylabel("$e_{orth}(r)$, Eq. (41)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout()
cone = rm.generators

## 3. PGA — Algorithm 1

The point of §3: replace the parameter-dependent `S_R(µ)` by one
parameter-independent `S_R^red`, computable entirely offline.

In [ ]:
wh = Whitener(hf.K, None)          # V carries the energy inner product
Ut = wh.L @ U
B_train = [wh.B_hat(hf.B(mu)) for mu in D_train]
B_valid = [wh.B_hat(hf.B(mu)) for mu in D_valid]

V_N = pod(Ut, delta=1e-4)
S_full = S_R_full(B_train, cone)
print(f"N = {V_N.shape[1]},  R = {cone.shape[1]},  dim S_R = {S_full.shape[1]}")
for d in (0.9, 0.7, 0.5):
    out = pga(B_train, V_N, cone, delta=d)
    print(f"  delta_PGA = {d}  ->  dim S_R^red = {out.S.shape[1]:3d}"
          f"   ({S_full.shape[1]/out.S.shape[1]:.1f}x smaller)")

In [ ]:
out = pga(B_train, V_N, cone, delta=0.5)
s = np.array(out.sigmas)
print(f"Lemma 3.2 — sigma non-increasing: {bool(np.all(np.diff(s) <= 1e-12))}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(s, "o-", ms=3)
ax.set_xlabel("n"); ax.set_ylabel(r"$\sigma_{S^n}(\mu_n)$")
ax.set_title("PGA greedy residual (Lemma 3.2)"); ax.grid(alpha=.3)
plt.tight_layout()

## 4. Proposition 3.1 — the theorem everything rests on

If `σ_S(µ) < β^on/c_S` (Eq. 23), then `β*_S = (β^on − c_S σ_S)/(1 + σ_S)` is a
**positive lower bound** on the inf-sup constant of the enriched pair (Eq. 24).

In [ ]:
V_off = orth_union(V_N, out.S)          # Eq. (29)
ok = appl = 0
for Bh in B_valid:
    Z = supremizer_space(Bh, cone)
    sig = sigma_S(Z, V_off)[0]
    c_S = boundedness_c_S(Bh, V_off, cone)
    b_on = inf_sup(Bh, orth_union(V_N, Z), cone)
    if sig < b_on / c_S:                                # Eq. (23)
        appl += 1
        beta_star = (b_on - c_S*sig) / (1 + sig)         # Eq. (24)
        b_off = inf_sup(Bh, V_off, cone)                 # Eq. (30)
        ok += (beta_star > 0) and (b_off >= beta_star - 1e-9)
print(f"criterion (23) applies for {appl}/{len(B_valid)} mu")
print(f"bound (24) valid in {ok}/{appl} of those")

## 5. Why `β^dec ≈ 0` does *not* reproduce here

§5.1 says `N < R` makes the decorrelated pair unstable. It doesn't here — and the
reason is that the inf in Eq. (14) runs over a **cone**, not a subspace:

$$\beta^{dec} = \min_{\alpha \ge 0,\ \alpha \ne 0} \frac{\|C\alpha\|}{\|X\alpha\|}, \qquad C = Q^T \hat B^T X$$

so `β^dec = 0` iff `ker(C)` **meets the non-negative orthant**. `N < R` gives
`dim ker(C) = R − N > 0`, which is necessary but not sufficient.

In [ ]:
V_small = pod(Ut, delta=1e-2)
Bh = B_valid[0]
C = V_small.T @ Bh.T @ cone
n = C.shape[1]
print(f"N = {V_small.shape[1]} < R = {n};  rank(C) = {np.linalg.matrix_rank(C)},"
      f"  dim ker(C) = {n - np.linalg.matrix_rank(C)}")

# Is there a non-negative kernel vector?
lp = linprog(np.zeros(n), A_eq=np.vstack([C, np.ones(n)]),
             b_eq=np.concatenate([np.zeros(C.shape[0]), [1.0]]),
             bounds=[(0, None)]*n, method="highs")
print(f"exists alpha >= 0 with C alpha = 0 ?  {lp.status == 0}")

# Gordan's theorem: a certificate that there is not.
m = C.shape[0]
g = linprog(np.concatenate([np.zeros(m), [-1.0]]),
            A_ub=np.hstack([-C.T, np.ones((n, 1))]), b_ub=np.zeros(n),
            bounds=[(-1, 1)]*m + [(0, None)], method="highs")
print(f"Gordan certificate: max t with C^T y >= t = {-g.fun:.4f}")
print("t > 0  =>  C^T y > 0 componentwise  =>  no non-negative kernel vector")
print(f"\nmeasured beta^dec = {inf_sup(Bh, V_small, cone):.4f}  (genuinely positive)")

With a single lower obstacle every constraint row is `−φ(s_i)` with `φ ≥ 0`, so
every supremizer lies in one halfspace and no non-negative combination can cancel.
`ObstacleHF(channel_gap=...)` adds a two-sided constraint whose blocks have
opposite signs; it produces genuine two-sided activity but still did not drive
`β^dec` to zero at any `(N,R)` tried. Reproducing the instability appears to need
the 2-D contact geometry of §5.2.

This is a property of the test problem, not a defect in the algorithms — which is
why §4 above tests Proposition 3.1 directly instead.